# TQC Reproduction: Ant-v4 Benchmark Training (ICML 2020) - Phase 11

Autonomous Kaggle GPU reproduction notebook for **Truncated Quantile Critics (TQC)** on **Ant-v4**.
- **Paper Reference**: Arsenii Kuznetsov et al. *"Controlling Overestimation Bias with Truncated Mixture of Continuous Distributional Quantile Critics"* (ICML 2020).
- **Published Target Score**: **6,293 ± 276 (ICML 2020 Table 1) / 8.01 ± 0.87k (Table 2)** (Table 1 / Table 2)
- **Critic Configuration**: $M=5$ critics, $N=25$ quantiles ($MN=125$ atoms), $d=5$ top quantiles discarded
- **Default Multi-Seed Suite**: Seeds `[42, 43, 44, 45, 46]` (1,000,000 steps each)


In [ ]:
# 1. Verify GPU Accelerator and Configure Headless EGL Rendering
import os
import sys
import subprocess

os.environ["MUJOCO_GL"] = "egl"

import torch
print('=== Hardware Environment ===')
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device      : {gpu_name} ({vram:.2f} GB VRAM)')
    torch.backends.cudnn.benchmark = True
else:
    print('WARNING: Running without GPU accelerator. In Kaggle, enable GPU under Notebook Options -> Accelerator -> GPU T4 x 2 or P100.')


In [ ]:
# 2. Install Gymnasium MuJoCo Simulation Harness and Rendering Libraries
!pip install -q "gymnasium[mujoco]>=0.29.0" "mujoco>=3.0.0" imageio imageio-ffmpeg pyyaml

import gymnasium as gym
test_env = gym.make('Ant-v4')
obs, _ = test_env.reset()
print(f'Environment Ant-v4 operational. State shape: {obs.shape}, Action shape: {test_env.action_space.shape}')
test_env.close()


In [ ]:
# 3. Setup Project Repository in Kaggle Workspace
import os
import sys
import base64
import io
import tarfile

REPO_DIR = '/kaggle/working/RL_project'
os.makedirs(REPO_DIR, exist_ok=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Unpack embedded complete project codebase
b64_payload = """..."""
if b64_payload:
    tar_bytes = base64.b64decode(b64_payload.strip())
    with tarfile.open(fileobj=io.BytesIO(tar_bytes), mode='r:gz') as tar:
        tar.extractall(path='.')
    print('Successfully unpacked embedded TQC project codebase into workspace.')
else:
    !git clone https://github.com/aryanthepain/RL_project.git {REPO_DIR}

print(f'Working directory set to: {os.getcwd()}')


In [ ]:
# 4. Configure Benchmark Parameters
ENV_ID = 'Ant-v4'
SEEDS = [42, 43, 44, 45, 46]  # Multi-seed evaluation suite
TOTAL_STEPS = 1_000_000        # Standard 1M benchmark steps
EVAL_FREQ = 5_000              # Evaluation every 5k steps
EVAL_EPISODES = 10             # 10 deterministic rollouts per evaluation
CHECKPOINT_FREQ = 50_000       # Periodic checkpoint saving
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LOG_DIR = f'runs/Ant-v4'
RESULTS_DIR = f'results/Ant-v4'

print(f'Configuration ready for {ENV_ID}: {len(SEEDS)} seeds x {TOTAL_STEPS:,} steps on {DEVICE}')


In [ ]:
# 5. Execute Phase 11 Benchmark Training & Bias Diagnostics
# Set --seeds to 42 for a single-seed pilot run, or --seeds 42 43 44 45 46 for the full suite
!python scripts/run_ant_benchmark.py \
    --seeds 42 \
    --steps 1000000 \
    --eval-freq 5000 \
    --eval-episodes 10 \
    --checkpoint-freq 50000 \
    --log-dir runs/Ant-v4 \
    --results-dir results/Ant-v4 \
    --mode full


In [ ]:
# 6. Display Replication Learning Curves and Diagnostics
from IPython.display import Image, Markdown, display
import os

curves_path = f'results/Ant-v4/ant_reproduction_learning_curves.png'
bias_plot_path = f'results/Ant-v4/ant_bias_diagnostics.png'
table_path = f'results/Ant-v4/ant_results_table.md'

if os.path.isfile(curves_path):
    print('Replication Learning Curves:')
    display(Image(curves_path))

if os.path.isfile(bias_plot_path):
    print('Overestimation Bias Diagnostics:')
    display(Image(bias_plot_path))

if os.path.isfile(table_path):
    with open(table_path, 'r', encoding='utf-8') as f:
        display(Markdown(f.read()))


In [ ]:
# 7. Render Evaluation Video with Adaptive Inactivity Truncation
import glob
from IPython.display import Video, display

ckpts = sorted(glob.glob(f'runs/Ant-v4/**/best_model.pt', recursive=True))
if ckpts:
    best_ckpt = ckpts[0]
    video_out = f'/kaggle/working/ant_eval.mp4'
    print(f'Rendering evaluation video from: {best_ckpt}')
    !python src/tqc/visualize.py --checkpoint {best_ckpt} --env Ant-v4 --output {video_out} --truncate-inactive --episodes 1
    if os.path.isfile(video_out):
        display(Video(video_out, embed=True, width=640))
else:
    print('No checkpoints found to render video.')


In [ ]:
# 8. Package Artifacts into .tar.gz with SHA256 Verification
import hashlib
import tarfile

archive_name = '/kaggle/working/tqc_ant_phase11_artifacts.tar.gz'
target_run_dir = 'runs/Ant-v4'

if os.path.isdir(target_run_dir):
    print(f'Creating compressed artifact archive: {archive_name}')
    with tarfile.open(archive_name, 'w:gz') as tar:
        tar.add(target_run_dir, arcname='Ant-v4')
        if os.path.isdir('results/Ant-v4'):
            tar.add('results/Ant-v4', arcname='results_ant')

    h = hashlib.sha256()
    with open(archive_name, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    sha256 = h.hexdigest()
    print(f'Artifact package ready ({os.path.getsize(archive_name) / (1024*1024):.2f} MB)')
    print(f'SHA256: {sha256}')
    with open('/kaggle/working/checksums_ant.sha256', 'w') as f:
        f.write(f'{sha256}  {os.path.basename(archive_name)}\n')
    print('Done! Artifacts are available in Kaggle Output for one-click download or API sync.')
else:
    print('Run directory runs/Ant-v4 not found.')
